# Batch-effect bifurcation -- distribution estimation + trajectories, seeds 0-9

Ten seeds of the bifurcation simulation (`sim.make_bifurcation(seed, strength=0.5, n_per_time=3000,
dim=10)`). Section 1 estimates each seed's series and exports the shared per-seed data npz every
baseline fits on (identical data across methods); Section 2 fits our trajectory methods (UOT maps /
flow OT-CFM unshared / flow UOT-map unshared) on the cached estimates; the metrics cell aggregates.
`SMOKE = 1` runs a single seed end to end; `SMOKE = 0` runs all ten at the paper's budgets.

In [ ]:
import os, sys, time, json
import numpy as np
import matplotlib.pyplot as plt


def _bootstrap():
    here = os.path.abspath(os.path.dirname(__file__)) if "__file__" in globals() else os.getcwd()
    # this folder (its helper modules) + tools/ (the shared `_repo.py` path resolver)
    for d in (here, os.path.abspath(os.path.join(here, os.pardir, os.pardir, "tools"))):
        if d not in sys.path:
            sys.path.insert(0, d)
    import _repo
    _repo.add_paths()
    return _repo


P = _bootstrap()
REPO = P.REPO
import uotreg as U
from uotreg import simulation as sim, trajectory_metrics as TM
from uotreg.metrics import w2
from uotreg.plotting import show_noisy_data, show_estimates, plot_trajectory_panel
import _be_common as C   # shared data npz path/format for the baseline files

## Shared parameters
`EST_DEFAULT` / `K_DEFAULT` are the paper configs; per-seed overrides go in `EST_OVERRIDES`.
The reported flows use an UNSHARED field (one v(x,t) per interval -- a shared field must average
the bimodal targets at the split); `flow_iters` / `flow_batch` are the knobs to tune.

In [ ]:
DIM        = globals().get("DIM", 10)
# ----------------------------------------------------------------------------- SMOKE
# 1 = small and fast: runs end to end on a laptop. **NOT the paper's numbers.**
# 0 = the settings used in the paper.
SMOKE      = 1
# 1 = write results/figures to `new_results/`; 0 = keep everything in memory.
# The shipped `results/` tree is never modified either way.
SAVE = globals().get("SAVE", 0)
# The estimated clouds, trajectories and metrics are a few MB per seed; the trained generators
# (~20 MB per seed) are needed by nothing downstream, so they are skipped unless SAVE_HEAVY = 1.
SAVE_HEAVY = 0
DEVICE     = globals().get("DEVICE", "cpu")
STRENGTH   = 0.5
N_PER_TIME = 200 if SMOKE else 3000        # cells per snapshot (== old file / exporter)
TRAJ_T     = list(range(1, 9))             # times 1..8 (split onset ~t=4)
STD_SPLIT  = "none" if DIM == 2 else {4: "std", 5: "std", 6: "std", 7: "std", 8: "std", "default": "std"}
RESULTS    = P.results("batcheffectnew")                       # read
RESULTS_W  = P.results("batcheffectnew", write=True)           # write (only when SAVE)
GENDIR     = os.path.join(RESULTS, "generators")               # read
GENDIR_W   = os.path.join(RESULTS_W, "generators")             # write
TAG        = f"bifurcation_d{DIM}_new" + ("_quick" if SMOKE else "")
SEEDS      = [0] if SMOKE else list(range(10))   # SMOKE: one seed runs end to end
METHODS    = ["ours: UOT maps",                  # composed UOT maps (the paper's trajectory method)
              "ours: flow OT-CFM unshared",      # flow, minibatch-OT coupling, field per interval
              "ours: flow UOT-map unshared"]     # flow trained against OUR maps (chain reused, fit once)
MK = [("adher_labeled", "adher*"), ("adher_indiv", "indiv"), ("w2_path", "w2_path"),
      ("spread_post", "spread"), ("roughness", "rough"), ("balanceA", "balA")]

EST_DEFAULT = dict(h=1.0, tau=5.0, budget=(12 if SMOKE else 100), std_mode=STD_SPLIT,
                   gen_hidden=100, gen_layers=4,
                   d_iters=(15 if SMOKE else 40), t_iters=(5 if SMOKE else 20), g_iters=(15 if SMOKE else 50),
                   n_gen=(200 if SMOKE else 3000), n_start=(40 if SMOKE else 120))
K_DEFAULT   = dict(traj_hidden=(64 if SMOKE else 256), map_layers=5,          # UOT maps
                   d_iters=(20 if SMOKE else 250), t_iters=(5 if SMOKE else 100),
                   flow_hidden=(64 if SMOKE else 128), flow_layers=4,         # flows (hidden 128)
                   flow_iters=(300 if SMOKE else 3000),                       # <- tunable
                   flow_batch=(64 if SMOKE else 256),                         # <- tunable
                   flow_lr=1e-3, flow_sigma=0.0, flow_n_per=20, flow_field="mlp", flow_seed=0,
                   tau=5.0, uot_seed=0)                                       # UOT maps reproducible

EST, TRAJS, ROWS = {}, {}, {}      # seed -> estimate dict / trajectories / metric rows

_gen_path = lambda seed, t, write=False: os.path.join(GENDIR_W if write else GENDIR,
                                                     f"{TAG}_seed{seed}_G_t{t}.pth")
_musd_path = lambda seed, write=False: os.path.join(GENDIR_W if write else GENDIR,
                                                   f"{TAG}_seed{seed}_musd.npy")


def _musd(observed, mode, dim):
    """The pooled mean/std `U.estimate` standardizes by -- needed to un-normalize a saved generator's
    draws back to the original coordinates (`std_mode` is per-time, so this is stored per time)."""
    if mode in ("std", "iso"):
        pool = np.concatenate([np.asarray(o, np.float32) for o in observed], 0)
        mu, sd = pool.mean(0), pool.std(0) + 1e-6
        if mode == "iso":
            sd = (pool.std(0).mean() + 1e-6) * np.ones(dim, np.float32)
        return np.asarray(mu, np.float32), np.asarray(sd, np.float32)
    return np.zeros(dim, np.float32), np.ones(dim, np.float32)


def estimate_seed(seed, cfg, viz=True, save=None):
    """Section-1: generate the seed's data, export the shared baseline npz, estimate the series."""
    save = SAVE if save is None else save
    G = sim.make_bifurcation(seed=seed, strength=STRENGTH, n_per_time=N_PER_TIME, dim=DIM)
    # export the shared per-seed data (baselines load this -> identical data)
    raw = [np.asarray(G.observed[t], np.float32) for t in TRAJ_T]
    times = np.array([float(G.tlist[t]) for t in TRAJ_T], np.float32)
    X0 = raw[0][:cfg["n_start"]]
    labels0 = (np.asarray(G.labels[TRAJ_T[0]])[:cfg["n_start"]] if G.labels is not None else np.array([]))
    np.savez(C.data_path(DIM, seed, SMOKE, write=True), raw_series=np.stack(raw), times=times, X0=X0,
             labels0=labels0, dim=DIM, seed=seed, strength=STRENGTH, n_per=N_PER_TIME, n_start=cfg["n_start"])
    # estimate at every trajectory time (inner loops explicit; run_stage1's anchored series by hand)
    t0 = time.time(); est, musd = [], []
    for t in TRAJ_T:
        mode = U.resolve_std(cfg["std_mode"], t)
        s, e = U.estimate(G.observed, G.tlist, query_time=t, dim=DIM, h=cfg["h"], tau=cfg["tau"],
                          budget=cfg["budget"], std_mode=mode,
                          gen_hidden=cfg["gen_hidden"], gen_layers=cfg["gen_layers"],
                          d_iters=cfg["d_iters"], t_iters=cfg["t_iters"], g_iters=cfg["g_iters"],
                          n_gen=cfg["n_gen"], seed=seed, device=DEVICE, return_estimator=True)
        est.append(np.asarray(s)); musd.append(np.stack(_musd(G.observed, mode, DIM)))
        if save and SAVE_HEAVY:                       # the trained generator (~2.5 MB per time)
            os.makedirs(GENDIR_W, exist_ok=True)
            e.save(_gen_path(seed, t, write=True))          # keep the trained generator -> re-draw any N later
    s1 = dict(est_series=est, ours_series=[raw[0]] + est[1:], raw_series=raw,
              X0=X0, labels0=(labels0 if labels0.size else None))
    EST[seed] = dict(G=G, s1=s1, cfg=cfg)
    if save:
        os.makedirs(GENDIR_W, exist_ok=True)
        np.save(_musd_path(seed, write=True), np.stack(musd))                          # (T, 2, dim)
        np.savez(os.path.join(RESULTS_W, f"{TAG}_seed{seed}_est.npz"),
                 est_series=np.stack(est),                                 # (T, n_gen, dim)
                 raw_series=np.stack(raw),                                 # (T, n_per, dim)
                 X0=X0, labels0=labels0, times=times, musd=np.stack(musd),
                 n_gen=cfg["n_gen"], n_start=cfg["n_start"], dim=DIM, seed=seed,
                 cfg=json.dumps({k: v for k, v in cfg.items() if k != "std_mode"}))
    print(f"[seed {seed}] estimate ({time.time()-t0:.0f}s)  per-time W2 est|raw:")
    for i, t in enumerate(TRAJ_T):
        tr = np.asarray(G.truth(t)); we, wr = w2(est[i], tr), w2(raw[i], tr)
        print(f"   t={t}: {we:6.3f} | {wr:6.3f}" + ("  <-- est WORSE" if we > wr + 1e-6 else ""))
    if save:
        print(f"   saved {TAG}_seed{seed}_est.npz -> {os.path.relpath(RESULTS_W, P.REPO)}")
    if viz:
        show_noisy_data(G, TRAJ_T); plt.suptitle(f"seed {seed}: observed vs truth", y=1.02); plt.show()
        show_estimates(G, est, TRAJ_T, w2_fn=w2); plt.suptitle(f"seed {seed}: estimate vs truth", y=1.02); plt.show()
    return s1


def fit_seed(seed, K, viz=True):
    """Section-2: fit the three "ours" methods on the seed's cached estimate, score, viz."""
    save = SAVE
    assert seed in EST, f"run estimate_seed({seed}) first"
    G, s1 = EST[seed]["G"], EST[seed]["s1"]
    t0 = time.time()
    models = U.fit_trajectories(G, s1["ours_series"], s1["raw_series"], None, TRAJ_T, K,
                                methods=METHODS, device=DEVICE, return_models=True)
    trajs = {m: models[m](s1["X0"]) for m in METHODS}        # (T, N, 2) projected
    TRAJS[seed] = trajs
    ROWS[seed] = TM.report(G, trajs, TRAJ_T, s1["labels0"])
    print(f"[seed {seed}] fit {len(METHODS)} methods ({time.time()-t0:.0f}s)")
    # save per-seed trajectories + metrics
    if not save:
        print(f"[seed {seed}] SAVE=0 -- trajectories kept in memory only")
        return trajs
    os.makedirs(RESULTS_W, exist_ok=True)
    np.savez(os.path.join(RESULTS_W, f"{TAG}_seed{seed}_trajs.npz"),
             **{m.replace(' ', '_').replace(':', ''): np.asarray(t) for m, t in trajs.items()},
             labels0=(s1["labels0"] if s1["labels0"] is not None else np.array([])))
    with open(os.path.join(RESULTS_W, f"{TAG}_seed{seed}_metrics.json"), "w") as f:
        json.dump({m: {k: (None if isinstance(v, float) and np.isnan(v) else float(v))
                       for k, v in r.items()} for m, r in ROWS[seed].items()}, f, indent=2)
    if viz:
        plot_trajectory_panel(G, trajs, max_cells=60, title=f"[bifurcation] d={DIM} seed {seed}"); plt.show()
    return ROWS[seed]


print(f"[NEW bifurcation d={DIM}] SMOKE={SMOKE} device={DEVICE} n_per_time={N_PER_TIME}  TAG={TAG}")
print(f"  save -> {os.path.relpath(RESULTS_W, P.REPO)} (when SAVE=1)  METHODS={METHODS}")

## Section 1: distribution estimation
Each seed's fit saves the estimated clouds + generators and exports the shared baseline data npz.
Re-running a seed overwrites that seed's files only.

In [ ]:
# Per-seed overrides of the settled run (seed 1 needed a smaller bandwidth + budget); applied only
# at SMOKE=0 so the fast pass keeps its tiny budgets.
EST_OVERRIDES = {1: dict(h=0.75, budget=70)}

for k in SEEDS:
    estimate_seed(k, dict(EST_DEFAULT, **({} if SMOKE else EST_OVERRIDES.get(k, {}))))

## Section 2: trajectory fitting
Fits on the seed's cached estimate (run its Section 1 first, or use the reload cell below).

In [ ]:
for k in SEEDS:
    fit_seed(k, dict(K_DEFAULT))

## Metrics: per-seed + aggregate

In [ ]:
seeds = sorted(ROWS)
print(f"[bifurcation d={DIM}] fitted seeds: {seeds}\n")
for s in seeds:                                  # per-seed tables
    print(f"--- seed {s} ---")
    print("   " + f"{'method':28s} " + " ".join(f"{h:>8}" for _, h in MK))
    for m in METHODS:
        print("   " + f"{m:28s} " + " ".join(
            ("     n/a" if (isinstance(ROWS[s][m][k], float) and np.isnan(ROWS[s][m][k])) else f"{ROWS[s][m][k]:8.3f}")
            for k, _ in MK))
    print()

agg = {}
print(f"[bifurcation d={DIM}] AGGREGATE over {len(seeds)} seeds {seeds} (mean +/- std):")
print("   " + f"{'method':28s} " + " ".join(f"{h:>14}" for _, h in MK))
for m in METHODS:
    agg[m] = {}
    row = []
    for k, _ in MK:
        vals = [ROWS[s][m][k] for s in seeds
                if not (isinstance(ROWS[s][m][k], float) and np.isnan(ROWS[s][m][k]))]
        mu, sd = (float(np.mean(vals)), float(np.std(vals))) if vals else (float("nan"), 0.0)
        agg[m][k] = {"mean": mu, "std": sd, "n": len(vals)}
        row.append(f"{mu:6.3f}+-{sd:<5.3f}")
    print("   " + f"{m:28s} " + " ".join(row))

if SAVE:
    os.makedirs(RESULTS_W, exist_ok=True)
    with open(os.path.join(RESULTS_W, f"{TAG}_agg_metrics.json"), "w") as f:
        json.dump({"seeds": seeds, "metrics": agg}, f, indent=2)
    print(f"\nsaved {TAG}_agg_metrics.json -> {os.path.relpath(RESULTS_W, P.REPO)}")